# Local wfield processing template

Use this notebook as a checklist for local motion correction, SVD/hemodynamic correction, Allen alignment, cue-aligned maps, and lick-aligned maps. Run from the `wfield` conda environment.

In [ ]:
from pathlib import Path
import json

repo = Path(r"C:/Github/Widefield_DAQ_recorder")
session = Path(r"E:/labcams_data/20260601/PS95_20260601_153653")
daq_h5 = Path(r"E:/DAQ_recorder_output/PS95_baseline_20260601_153627.h5")
camlog = session / "pco_edge_run000_00000000.camlog"
motion = session / "motion_corrected"
results = motion / "wfield_local_results"
allen = results / "allen_aligned_v6"
landmarks = session / "raw_widefield_data" / "dorsal_cortex_landmarks.json"
print(session)
print(daq_h5)
print(camlog)


## Motion correction

Run this in PowerShell if motion correction has not already been done.

In [ ]:
print(fr'''conda activate wfield
cd "{repo}"
python .\wfield_local\run_wfield_motion.py \
  "{session / 'raw_widefield_data' / 'pco_edge_run000_00000000_2_540_640_uint16.dat'}" \
  --output-dir "{motion}" \
  --mode 2d''')


## SVD and hemodynamic correction

`--functional-channel 1` is correct for this rig: channel 0 is 415 nm and channel 1 is 470 nm.

In [ ]:
print(fr'''python .\wfield_local\run_wfield_local.py \
  "{motion / 'motioncorrect_2_540_640_uint16.bin'}" \
  --output-dir "{results}" \
  --functional-channel 1 \
  --n-components 100''')


## Allen alignment

In [ ]:
print(fr'''python .\wfield_local\apply_allen_transform.py \
  "{results}" \
  --landmarks "{landmarks}" \
  --output "{allen}"''')


## Cue-aligned spout-position maps

In [ ]:
spout_out = motion / "spout_trial_averages_allen_v6"
print(fr'''python .\wfield_local\plot_spout_trial_averages.py \
  --label PS95_v6 \
  --daq-h5 "{daq_h5}" \
  --wfield-results "{results}" \
  --allen-dir "{allen}" \
  --camlog "{camlog}" \
  --output "{spout_out}" \
  --frame-align pco \
  --pre-s 1.0 \
  --post-s 1.0 \
  --fs 31.23''')


## Post-lick maps

In [ ]:
lick_out = motion / "lick_aligned_v6"
print(fr'''python .\wfield_local\plot_lick_aligned_averages.py \
  --label PS95_v6 \
  --daq-h5 "{daq_h5}" \
  --wfield-results "{results}" \
  --allen-dir "{allen}" \
  --camlog "{camlog}" \
  --output "{lick_out}" \
  --frame-align pco \
  --lick-thresh-upper-v 2.5 \
  --lick-thresh-lower-v 1.0 \
  --refractory-s 0.10 \
  --post-s 0.150 \
  --fs 31.23''')


## Inspect a summary JSON

In [ ]:
summary = lick_out / "PS95_v6_lick_aligned_150ms_post_by_spout_summary.json"
if summary.exists():
    payload = json.loads(summary.read_text())
    print(json.dumps(payload, indent=2))
else:
    print(f"Not found yet: {summary}")
